# SolverV8 example: multi-pathway 2Q spectrum

Notebook version of `chi7_multiorder_plot_example.py`.

The current configuration uses five interactions (`--+++`), so it computes the chi5-style 2Q example even though the original script name contains chi7.


In [1]:
from pathlib import Path

import numpy as np
import time
import pickle
import sys
from tqdm.notebook import tqdm

from SolverV9 import (
    LiouvilleSpectroscopySolver,
    SpectroscopyPlotter,
    standard_nq_protocol,
)

np.set_printoptions(threshold=sys.maxsize, linewidth=190, precision=5, suppress=True)

### Coupled Model

### Solver Parameters

- `T`: System temperature. `T = 0` assumes that the system initially occupies its ground state.
- `Eta`: Homogeneous broadening used in the resolvents, in eV. It primarily controls the spectral linewidth.
- `backend`: Numerical implementation used by the solver. `"dense"` uses dense matrices and NumPy linear-algebra routines.

In [2]:
solver_params = {
    "T": 0.0,
    "Eta": 0.005,
    "backend": "dense",
    "parallel_backend": "threading",
    "n_jobs": -1,
}

In [3]:
hbar = 0.658211951  # in eV fs
kB = 8.617333262 * 1e-5  # Boltzmann constant eV/K
T = 300  # temperature in K
kT = T * kB
beta = 1 / kT

# parameters
scale = 45
scale_en = 5

en_b=1.4907 # bright state excitonic resonant energy in eV
en_d=1.4907 # dark state excitonic resonant energy in eV
detuning = -0.004 * scale_en # detuning between exciton and cavity in eV
en_cav=en_b + detuning # cavity resonant energy in eV

N = 2  # maximum quantum
order = 3

rabi = 0.003 * scale  # vaccum rabi splitting
g = rabi / 2  # coupling strength for light-matter interaction in eV
J = 0.00075 * scale  # coupling strength between heavy and light excitons (from small band gap energy difference)
n_th = 0. # average number of cavity photons

muc = 1.0  # dipole strength for cavity photons
mub = 1.0  # dipole strength for heavy excitons
mud = 0.0  # dipole strength for light excitons

mu_qm1 = 0.3  # dipole strength for C
mu_qm2 = 0.29  # dipole strength for CC
mu_d1 = 0.04  # dipole strength for D
mu_d2 = 0.25  # dipole strength for DD

kappa = 0.0011 * scale  # cavity decay strength
gamma_b_decay = 0.0005 * scale  # heavy exciton decay strength
gamma_d_decay = 0.0001 * scale  # light exciton decay strength
gamma_cav_phase = 0.0006 * scale  # cavity dephasing strength
gamma_exc_phase = 0.0001 * scale  # exciton dephasing strength
gamma_bd = 0.001  # exciton species swapping strength

bb_bind=0.0035 * scale_en # strength of BB biexciton binding
bd_bind=-0.001 * scale_en # strength of BD biexciton binding
dd_bind=-0.001 * scale_en # strength of DD biexciton binding

model = "rw"  # "no_rw" for no rotating wave approximation, "rw" for rotating wave approximation
# model="no-rw"

date = time.strftime('%Y-%m-%d')
hour = time.strftime("%Hh%Mm%S")

### 1 exciton maximum
essentially N=inf because of translation symmetry

In [4]:
if N == 1:

    # setting up dm
    exc_basis = np.zeros((4, 4))
    exc_basis[0, 0] = 1 # ground,B,C,D
    rho = exc_basis @ exc_basis.T # ground state of Hamiltonian

    wc = en_cav / hbar
    wb = en_b / hbar  # bright exciton resonant frequencies
    wd = en_d / hbar  # dark exciton resonant frequencies

    # each individual lowering operators
    b_low = np.zeros_like(exc_basis)
    b_low[0, 1] = 1

    c_low = np.zeros_like(exc_basis)
    c_low[0, 2] = 1

    d_low = np.zeros_like(exc_basis)
    d_low[0, 3] = 1

    cav_low = c_low
    exc_b_low = b_low
    exc_d_low = d_low

    H0 = hbar * (wb * b_low.T @ b_low + wc * c_low.T @ c_low + wd * d_low.T @ d_low)  # energies

    H_coupling = b_low.T @ d_low  # B-D coupling
    H_coupling += H_coupling.T

    if model == "no-rw":
        H_int = mub * b_low.T @ c_low + mud * c_low.T @ d_low  # 1Q light-matter coupling
    elif model == "rw":
        H_int = mub * b_low.T @ c_low + mud * c_low.T @ d_low  # 1Q light-matter coupling
    else:
        print('Wrong model name')
    H_int += H_int.T

### 0-quantum + 1-quantum + 2-quantum

In [5]:
if N == 2:

    # setting up dm
    exc_basis = np.zeros((10, 10))
    exc_basis[0, 0] = 1 # ground,B,C,D,BB,BC,BD,CC,CD,DD
    rho = exc_basis @ exc_basis.T  # ground state of Hamiltonian

    wc = en_cav / hbar
    wb = en_b / hbar  # bright exciton resonant frequencies
    wd = en_d / hbar  # dark exciton resonant frequencies

    # each individual lowering operators
    b_low = np.zeros_like(exc_basis)
    b_low[0, 1] = 1

    c_low = np.zeros_like(exc_basis)
    c_low[0, 2] = 1

    d_low = np.zeros_like(exc_basis)
    d_low[0, 3] = 1

    bb_low = np.zeros_like(exc_basis)
    bb_low[1, 4] = 1

    bc_low = np.zeros_like(exc_basis)
    bc_low[1, 5] = 1 / np.sqrt(2)

    cb_low = np.zeros_like(exc_basis)
    cb_low[2, 5] = 1 / np.sqrt(2)

    bd_low = np.zeros_like(exc_basis)
    bd_low[1, 6] = 1 / np.sqrt(2)

    db_low = np.zeros_like(exc_basis)
    db_low[3, 6] = 1 / np.sqrt(2)

    cc_low = np.zeros_like(exc_basis)
    cc_low[2, 7] = 1

    cd_low = np.zeros_like(exc_basis)
    cd_low[2, 8] = 1 / np.sqrt(2)

    dc_low = np.zeros_like(exc_basis)
    dc_low[3, 8] = 1 / np.sqrt(2)

    dd_low = np.zeros_like(exc_basis)
    dd_low[3, 9] = 1

    # global lowering operators
    cav_low = c_low + bc_low + cc_low + dc_low
    exc_b_low = b_low + bb_low + cb_low + db_low
    exc_d_low = d_low + bd_low + cd_low + dd_low

    H0_b = b_low.T @ b_low  # b-pop
    H0_c = c_low.T @ c_low  # c-pop
    H0_d = d_low.T @ d_low  # d-pop
    H0_bb = bb_low.T @ bb_low  # bb-pop
    H0_bc = bc_low.T @ bc_low + cb_low.T @ cb_low  # bc-pop
    H0_bd = bd_low.T @ bd_low + db_low.T @ db_low  # bd-pop
    H0_cc = cc_low.T @ cc_low  # cc-pop
    H0_cd = cd_low.T @ cd_low + dc_low.T @ dc_low  # cd-pop
    H0_dd = dd_low.T @ dd_low  # dd-pop

    H0 = hbar * (wb * H0_b + wc * H0_c + wd * H0_d + (2 * wb - bb_bind) * H0_bb + (wb + wc) * H0_bc + (
                wb + wd - bd_bind) * H0_bd + 2 * wc * H0_cc + (wc + wd) * H0_cd + (2 * wd - dd_bind) * H0_dd)

    H_coupling = b_low.T @ d_low  # 1Q B-D coupling
    H_coupling += 2 * (bb_low.T @ bd_low + cb_low.T @ cd_low + db_low.T @ dd_low)  # 2Q B-D coupling
    H_coupling += H_coupling.T

    H_int = mub * b_low.T @ c_low + mud * c_low.T @ d_low  # 1Q light-matter coupling
    H_int += mub * 2 * (
                bb_low.T @ bc_low + cb_low.T @ cc_low + db_low.T @ dc_low)  # 2Q bright light-matter coupling
    H_int += mud * 2 * (
                bc_low.T @ bd_low + cc_low.T @ cd_low + dc_low.T @ dd_low)  # 2Q dark light-matter coupling
    if model == "no-rw":
        H_int += 2 * cav_low * (mub * exc_b_low + mud * exc_d_low)  # 2Q non-RWA terms (don't conserve energy)
    elif model != "rw":
        print('Wrong model name')
    H_int += H_int.T
    # print(H_int)

In [6]:
H = H0 + hbar * g * H_int + J * H_coupling  # total hamiltonian
# print("H0", H0)
# print("H_int", H_int)
# print("H_coupling", H_coupling)
print("H", H)

ad = mu_qm2 * cav_low + (mu_qm1 - mu_qm1) * c_low + mu_d2 * exc_d_low + (mu_d1 - mu_d2) * d_low  # lowering operator
ad_paper = mu_qm2 * cav_low + mu_qm2 * exc_b_low  # would have been the lowering operator according to Giuseppe's model. Cavity through quasi-coupling and dark exciton through dipole interaction. That one doesn't work though so I chose one that does.
# obs = muc * (cav_low + cav_low.T) + mub * (exc_b_low + exc_b_low.T) + mud * (exc_d_low + exc_d_low.T) # observable.
obs = ad + ad.T
# print("mud", mud)
# print("ad", ad)

# collapse operators
c_cav_rel = np.sqrt(kappa * (n_th + 1)) * cav_low # cavity relaxation
c_cav_exc = np.sqrt(kappa * n_th) * cav_low.T # cavity excitation
c_mat_rel = np.sqrt(gamma_b_decay * (n_th + 1)) * exc_b_low + np.sqrt(gamma_d_decay * (n_th + 1)) * exc_d_low # exciton relaxation
c_mat_exc = np.sqrt(gamma_b_decay * n_th) * exc_b_low.T + np.sqrt(gamma_d_decay * n_th) * exc_d_low.T # exciton excitation

c_dep = (np.sqrt(gamma_exc_phase) * (wb * H0_b + wd * H0_d) + # B and D pop
        np.sqrt(gamma_cav_phase) * wc * H0_c + # C pop
        np.sqrt(gamma_exc_phase) * (2 * wb - bb_bind) * H0_bb + # BB pop
        (np.sqrt(gamma_cav_phase) + np.sqrt(gamma_exc_phase)) * (wb + wc) / 2 * H0_bc + # BC pop
        np.sqrt(gamma_exc_phase) * (wb + wd) * H0_bd + # BD pop
        np.sqrt(gamma_cav_phase) * 2 * wc * H0_cc + # CC pop
        (np.sqrt(gamma_cav_phase) + np.sqrt(gamma_exc_phase)) * (wc + wd) / 2 * H0_cd + # CD pop
        np.sqrt(gamma_exc_phase) * (2 * wd - dd_bind ) * H0_dd) # DD pop

c_b_d = np.sqrt(gamma_bd) * exc_d_low.T * exc_b_low # bright exciton to dark exciton transition
c_d_b = np.sqrt(gamma_bd) * exc_b_low.T * exc_d_low # dark exciton to bright exciton transition

c_b_cav = np.sqrt(gamma_bd) * cav_low.T * exc_b_low # bright exciton to dark exciton transition
c_cav_b = np.sqrt(gamma_bd) * exc_b_low.T * cav_low # dark exciton to bright exciton transition

c_ops = [c_cav_rel, c_cav_exc, c_dep, c_mat_rel, c_mat_exc, c_b_d, c_d_b, c_b_cav, c_cav_b]
# print(c_ops)

H [[0.      0.      0.      0.      0.      0.      0.      0.      0.      0.     ]
 [0.      1.4907  0.04443 0.03375 0.      0.      0.      0.      0.      0.     ]
 [0.      0.04443 1.4707  0.      0.      0.      0.      0.      0.      0.     ]
 [0.      0.03375 0.      1.4907  0.      0.      0.      0.      0.      0.     ]
 [0.      0.      0.      0.      2.96988 0.06283 0.04773 0.      0.      0.     ]
 [0.      0.      0.      0.      0.06283 2.9614  0.      0.06283 0.03375 0.     ]
 [0.      0.      0.      0.      0.04773 0.      2.98469 0.      0.04443 0.04773]
 [0.      0.      0.      0.      0.      0.06283 0.      2.9414  0.      0.     ]
 [0.      0.      0.      0.      0.      0.03375 0.04443 0.      2.9614  0.     ]
 [0.      0.      0.      0.      0.      0.      0.04773 0.      0.      2.98469]]


### solver

In [7]:
solver = LiouvilleSpectroscopySolver(solver_params)
solver.feed_model(
    H,
    obs,
    c_ops_raw=c_ops,
    initial_density_matrix=rho,
    density_matrix_basis="site",
)

print("Basis size:", len(exc_basis))


--- Model loading ---
Model transformed to the eigenbasis.
Liouville backend ready: dense.
Basis size: 10


### Example: Fifth-Order Double-Quantum Signal

This example generates the Liouville pathways contributing to a fifth-order (`χ⁽⁵⁾`) double-quantum (`2Q`) signal.

#### Pathway generation

- `"--+++"`: Phase-matching signature of the five field interactions. The first two interactions use the negative-frequency field component, while the last three use the positive-frequency component.
- `maximum_manifold=2`: Pathways may access states containing at most two excitations. This allows the doubly excited manifold required for a `2Q` coherence.
- `component="chi5_2q"`: Label assigned to this fifth-order double-quantum contribution.

#### Spectroscopy protocol

- `n_interactions=5`: Five light-matter interactions are applied, corresponding to a `χ⁽⁵⁾` response.
- `order=2`: Only pathways containing a second-order coherence, `|q| = 2`, are selected.
- `nq_interval=2`: The `2Q` coherence is required during the second evolution interval, after the second interaction. This interval is Fourier transformed along `omega_2q`.
- `detection_interval=5`: The emitted coherence is evaluated after the fifth and final interaction. This interval is Fourier transformed along `omega_emit`.
- `nq_axis="omega_2q"`: Name of the double-quantum frequency axis.
- `detection_axis="omega_emit"`: Name of the emission-frequency axis.


In [8]:
# pathways = solver.generate_pathways_with_ufss(
#     "--+++",
#     maximum_manifold=2,
#     component="chi5_2q",
# )
#
# protocol = standard_nq_protocol(
#     order=2,
#     nq_interval=2,
#     detection_interval=5,
#     n_interactions=5,
#     nq_axis="omega_1q",
#     detection_axis="omega_emit",
# )
#
# [(pathway.name, pathway.interactions, pathway.coherence_orders) for pathway in pathways]

pathways = solver.generate_pathways_with_ufss(
    "-++",
    maximum_manifold=2,
    component="chi3_2q",
)

protocol = standard_nq_protocol(
    order=1,
    nq_interval=1,
    detection_interval=3,
    n_interactions=3,
    nq_axis="omega_1q",
    detection_axis="omega_emit",
)

[(pathway.name, pathway.interactions, pathway.coherence_orders) for pathway in pathways]


[('R3', ('Bu', 'Ku', 'Ku'), (-1, 0, 1)),
 ('R1', ('Bu', 'Ku', 'Bd'), (-1, 0, 1)),
 ('R2', ('Bu', 'Bd', 'Ku'), (-1, 0, 1))]

In [9]:
omega_1q = np.linspace(-1.8, -1.0, 100)
omega_emit = np.linspace(1.0, 1.8, 100)
result = []
ran = 3
for i in tqdm(range(ran)):
    delays = {
        "t2": i*10
    }

    result.append(solver.generate_NQ_spectrum(
        1,
        protocol,
        axes={"omega_1q": omega_1q, "omega_emit": omega_emit},
        delays=delays,
        pathways=pathways)
    )

  0%|          | 0/3 [00:00<?, ?it/s]

Calculating 3 pathway spectrum/s on a 100x100 grid with protocol 'standard_1q' using parallel=threading, n_jobs=-1.
Using dense prefix-tree pathway reuse.
Calculating 3 pathway spectrum/s on a 100x100 grid with protocol 'standard_1q' using parallel=threading, n_jobs=-1.
Using dense prefix-tree pathway reuse.
Calculating 3 pathway spectrum/s on a 100x100 grid with protocol 'standard_1q' using parallel=threading, n_jobs=-1.
Using dense prefix-tree pathway reuse.


### Plotting Parameters

This block plots the individual `χ⁽⁵⁾` double-quantum pathways and their summed `2Q` signal.

- `save_pdf=True`: Saves the resulting figure as a PDF.
- `output_directory`: Directory where the PDF is written.
- `detection_phase=0`: Applies no additional phase rotation to the complex signal before selecting the displayed component.
- `result`: Spectrum result returned by `solver.generate_NQ_spectrum(...)`.
- `pathways="all"`: Plots every individual pathway stored in `result`.
- `totals=["2Q"]`: Adds a panel containing the sum of the selected `2Q` pathways.
- `view="real"`: Displays the real part of each complex spectrum.
- `normalization="individual"`: Normalizes every pathway and total panel independently. This emphasizes spectral shapes but removes relative amplitude information between panels.
- `axis_labels`: Defines custom labels for the double-quantum and emission-frequency axes.
- `include_diagrams=False`: Does not add double-sided Feynman diagrams to the saved figure.
- `display_diagrams=False`: Does not display the diagrams separately.
- `save_pdf=save_pdf`: Enables or disables PDF export according to the value of `save_pdf`.
- `output_directory=...`: Uses the selected output directory when saving is enabled.
- `spectrum_pdf_name`: Filename used for the exported spectrum.
- `show=True`: Displays the figure after it is generated.

The returned `plot_result` object contains the generated figure and related plotting information.

> **Important:** With `normalization="individual"`, two pathways with very different amplitudes may appear equally intense. Use a common or global normalization when relative pathway strengths must be compared. The user can also use ''none'' to dont normalized, or ''shared'' to normalized all the spectrum togheter. ''individual'' do a individual normalization of each spectrum.

In [10]:
save_pdf = True
output_directory = Path.cwd() / "Result_Test" / "Multiorder_pathway_plot" / date / hour
output_directory.mkdir(parents=True, exist_ok=True)

for i in tqdm(range(ran)):
    plotter = SpectroscopyPlotter(detection_phase=np.pi/2)
    plot_result = plotter.plot_pathways_multiorder(
        result[i],
        pathways="all",
        totals=["1Q"],
        view="real",
        normalization="none",
        axis_labels={
            "omega_1q": "1Q energy (eV)",
            "omega_emit": "Emission energy (eV)",
        },
        include_diagrams=False,
        display_diagrams=False,
        save_pdf=save_pdf,
        output_directory=output_directory if save_pdf else None,
        spectrum_pdf_name=f"chi3_Q1_real_t{i*10}.png",
        show=False,
    )
    
    plot_result = plotter.plot_pathways_multiorder(
        result[i],
        pathways="all",
        totals=["1Q"],
        view="imag",
        normalization="none",
        axis_labels={
            "omega_1q": "1Q energy (eV)",
            "omega_emit": "Emission energy (eV)",
        },
        include_diagrams=False,
        display_diagrams=False,
        save_pdf=save_pdf,
        output_directory=output_directory if save_pdf else None,
        spectrum_pdf_name=f"chi3_Q1_imag_t{i*10}.png",
        show=False,
    )

    plot_result = plotter.plot_pathways_multiorder(
        result[i],
        pathways="all",
        totals=["1Q"],
        view="abs",
        normalization="none",
        axis_labels={
            "omega_1q": "1Q energy (eV)",
            "omega_emit": "Emission energy (eV)",
        },
        include_diagrams=False,
        display_diagrams=False,
        save_pdf=save_pdf,
        output_directory=output_directory if save_pdf else None,
        spectrum_pdf_name=f"chi3_Q1_abs_t{i*10}.png",
        show=False,
    )

  0%|          | 0/3 [00:00<?, ?it/s]

In [11]:
# print("Plotted panels:", plot_result.panel_names)
# print("Matching UFSS diagrams:", tuple(plot_result.diagrams))
# if save_pdf:
#     print("Spectrum PDF:", plot_result.spectrum_pdf)
#     print("Diagram PDFs:", plot_result.diagram_paths)
